In [2]:
# ============================================== CONFIG & USER DEFINED FUNCTIONS ======================================================= #

from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
from datetime import datetime
import uuid

# =========================================================
# CONFIG
# =========================================================
def get_config(environment="dev"):
    config = {
        "dev": {
            "raw_base_path": "Files/raw",
            "bronze_schema": "dbo",
            "silver_schema": "dbo",
            "gold_schema": "dbo"
        },
        "preprod": {
            "raw_base_path": "Files/raw",
            "bronze_schema": "dbo",
            "silver_schema": "dbo",
            "gold_schema": "dbo"
        },
        "prod": {
            "raw_base_path": "Files/raw",
            "bronze_schema": "dbo",
            "silver_schema": "dbo",
            "gold_schema": "dbo"
        }
    }
    return config[environment]

CONTROL_LOG_TABLE = "ETL.pl_control_config_log"
BAD_RECORDS_TABLE = "ETL.pl_bad_records_log"
WATERMARK_TABLE = "ETL.pl_watermark_log"

def generate_run_id():
    return str(uuid.uuid4())

# =========================================================
# INSERT ONLY ONCE AT STEP START
# =========================================================
def log_step_start(
    run_id,
    pipeline_name,
    pipeline_step_name,
    source_file_path,
    target_table_name,
    pipeline_start_time
):
    schema = StructType([
        StructField("run_id", StringType(), True),
        StructField("pipeline_name", StringType(), True),
        StructField("pipeline_step_name", StringType(), True),
        StructField("source_file_path", StringType(), True),
        StructField("target_table_name", StringType(), True),
        StructField("step_status", StringType(), True),
        StructField("pipeline_status", StringType(), True),
        StructField("record_count", IntegerType(), True),
        StructField("error_message", StringType(), True),
        StructField("step_start_time", TimestampType(), True),
        StructField("step_end_time", TimestampType(), True),
        StructField("pipeline_start_time", TimestampType(), True),
        StructField("pipeline_end_time", TimestampType(), True),
        StructField("created_at", TimestampType(), True),
        StructField("updated_at", TimestampType(), True)
    ])

    now = datetime.now()

    row = [(
        run_id,
        pipeline_name,
        pipeline_step_name,
        source_file_path,
        target_table_name,
        "STARTED",
        "STARTED",
        0,
        None,
        now,
        None,
        pipeline_start_time,
        None,
        now,
        now
    )]

    df = spark.createDataFrame(row, schema)
    df.write.format("delta").mode("append").saveAsTable(CONTROL_LOG_TABLE)

# =========================================================
# UPDATE SAME ROW AT STEP END
# =========================================================
def log_step_end(
    run_id,
    pipeline_name,
    pipeline_step_name,
    step_status,
    pipeline_status,
    record_count,
    error_message=None,
    pipeline_end_time=None
):
    error_message = (error_message or "").replace("'", " ")

    if pipeline_end_time is None:
        pipeline_endt = "NULL"
    else:
        pipeline_endt = f"TIMESTAMP('{pipeline_end_time}')"

    spark.sql(f"""
        UPDATE {CONTROL_LOG_TABLE}
        SET step_status = '{step_status}',
            pipeline_status = '{pipeline_status}',
            record_count = {record_count},
            error_message = '{error_message}',
            step_end_time = current_timestamp(),
            pipeline_end_time = {pipeline_endt},
            updated_at = current_timestamp()
        WHERE run_id = '{run_id}'
          AND pipeline_name = '{pipeline_name}'
          AND pipeline_step_name = '{pipeline_step_name}'
    """)

# =========================================================
# OPTIONAL: UPDATE FINAL PIPELINE STATUS FOR ALL ROWS
# =========================================================
def finalize_pipeline_run(run_id, pipeline_name, final_status):
    spark.sql(f"""
        UPDATE {CONTROL_LOG_TABLE}
        SET pipeline_status = '{final_status}',
            pipeline_end_time = current_timestamp(),
            updated_at = current_timestamp()
        WHERE run_id = '{run_id}'
          AND pipeline_name = '{pipeline_name}'
    """)

# =========================================================
# RESTART / SKIP LOGIC
# =========================================================
def should_run_step(run_id, pipeline_name, pipeline_step_name, rerun_mode="FULL"):
    if rerun_mode.upper() == "FULL":
        return True

    df = spark.sql(f"""
        SELECT step_status
        FROM {CONTROL_LOG_TABLE}
        WHERE run_id = '{run_id}'
          AND pipeline_name = '{pipeline_name}'
          AND pipeline_step_name = '{pipeline_step_name}'
        ORDER BY updated_at DESC
        LIMIT 1
    """)

    rows = df.collect()

    if len(rows) == 0:
        return True

    latest_status = rows[0]["step_status"]

    if rerun_mode.upper() == "FAILED_ONLY" and latest_status == "SUCCESS":
        return False

    return True

# =========================================================
# DATE PARSER
# =========================================================
def parse_multi_format_timestamp(col_name):
    return F.coalesce(
        F.to_timestamp(F.col(col_name), "yyyy-MM-dd"),
        F.to_timestamp(F.col(col_name), "yyyy-MM-dd HH:mm:ss"),
        F.to_timestamp(F.col(col_name), "dd/MM/yyyy"),
        F.to_timestamp(F.col(col_name), "MM/dd/yyyy"),
        F.to_timestamp(F.col(col_name), "yyyy/MM/dd"),
        F.to_timestamp(F.col(col_name), "dd-MM-yyyy"),
        F.to_timestamp(F.col(col_name))
    )

# =========================================================
# DELTA MERGE
# =========================================================
def merge_upsert(df, target_table_name, merge_condition):
    if spark.catalog.tableExists(target_table_name):
        delta_table = DeltaTable.forName(spark, target_table_name)
        (
            delta_table.alias("t")
            .merge(df.alias("s"), merge_condition)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        df.write.format("delta").mode("overwrite").saveAsTable(target_table_name)

# ============================================== CONFIG & USER DEFINED FUNCTIONS ======================================================= #

StatementMeta(, 2d49eeb2-2977-4e3d-9f06-806883a16afa, 4, Finished, Available, Finished, False)

In [3]:
from pyspark.sql import functions as F
from datetime import datetime

# =========================================================
# PARAMETERS
# =========================================================
environment = globals().get("environment", "dev")

default_run_id = "45e6d27f-77cb-49a0-b1ae-7eecf73012bb"
run_id = globals().get("run_id", default_run_id)

rerun_mode = globals().get("rerun_mode", "FULL")
pipeline_name = "pl_retail_analytics_medallion"
pipeline_start_time = globals().get("pipeline_start_time", datetime.now())

if run_id is None:
    raise Exception("run_id must be passed from Bronze/Silver")

print(f"environment = {environment}")
print(f"run_id = {run_id}")
print(f"rerun_mode = {rerun_mode}")

StatementMeta(, 2d49eeb2-2977-4e3d-9f06-806883a16afa, 5, Finished, Available, Finished, False)

environment = dev
run_id = 45e6d27f-77cb-49a0-b1ae-7eecf73012bb
rerun_mode = FULL


In [4]:
# =========================================================
# 1. DIM_CUSTOMER
# =========================================================

step_name = "gold_dim_customer"
source_table = "dbo.silver_customers"
target_table = "dbo.dim_customer"

if should_run_step(run_id, pipeline_name, step_name, rerun_mode):
    try:
        log_step_start(run_id, pipeline_name, step_name, source_table, target_table, pipeline_start_time)

        df = spark.read.table(source_table)

        dim_df = df.select(
            F.col("customer_id"),
            F.col("customer_name"),
            F.col("email"),
            F.col("customer_created_ts"),
            F.lit(True).alias("is_current"),
            F.current_timestamp().alias("effective_from"),
            F.lit(None).cast("timestamp").alias("effective_to")
        )

        merge_upsert(dim_df, target_table, "t.customer_id = s.customer_id")

        count = dim_df.count()

        log_step_end(run_id, pipeline_name, step_name, "SUCCESS", "IN_PROGRESS", count)

    except Exception as e:
        log_step_end(run_id, pipeline_name, step_name, "FAILED", "FAILED", 0, str(e), datetime.now())
        raise

StatementMeta(, 2d49eeb2-2977-4e3d-9f06-806883a16afa, 6, Finished, Available, Finished, False)

In [5]:
# =========================================================
# 2. DIM_PRODUCT
# =========================================================

step_name = "gold_dim_product"
source_table = "dbo.silver_products"
target_table = "dbo.dim_product"

if should_run_step(run_id, pipeline_name, step_name, rerun_mode):
    try:
        log_step_start(run_id, pipeline_name, step_name, source_table, target_table, pipeline_start_time)

        df = spark.read.table(source_table)

        dim_df = df.select(
            "product_id",
            "product_name",
            "category",
            "unit_price"
        )

        merge_upsert(dim_df, target_table, "t.product_id = s.product_id")

        count = dim_df.count()

        log_step_end(run_id, pipeline_name, step_name, "SUCCESS", "IN_PROGRESS", count)

    except Exception as e:
        log_step_end(run_id, pipeline_name, step_name, "FAILED", "FAILED", 0, str(e), datetime.now())
        raise

StatementMeta(, 2d49eeb2-2977-4e3d-9f06-806883a16afa, 7, Finished, Available, Finished, False)

In [6]:
# =========================================================
# 3. DIM_DATE
# =========================================================

step_name = "gold_dim_date"
target_table = "dbo.dim_date"

if should_run_step(run_id, pipeline_name, step_name, rerun_mode):
    try:
        log_step_start(run_id, pipeline_name, step_name, "derived", target_table, pipeline_start_time)

        dates = spark.read.table("dbo.silver_orders") \
            .select(F.to_date("order_ts").alias("date")).distinct()

        dim_df = dates.select(
            F.col("date"),
            F.year("date").alias("year"),
            F.month("date").alias("month"),
            F.dayofmonth("date").alias("day"),
            F.quarter("date").alias("quarter")
        )

        merge_upsert(dim_df, target_table, "t.date = s.date")

        count = dim_df.count()

        log_step_end(run_id, pipeline_name, step_name, "SUCCESS", "IN_PROGRESS", count)

    except Exception as e:
        log_step_end(run_id, pipeline_name, step_name, "FAILED", "FAILED", 0, str(e), datetime.now())
        raise

StatementMeta(, 2d49eeb2-2977-4e3d-9f06-806883a16afa, 8, Finished, Available, Finished, False)

In [7]:
# =========================================================
# 4. FACT_SALES
# =========================================================
step_name = "gold_fact_sales"
target_table = "dbo.fact_sales"

if should_run_step(run_id, pipeline_name, step_name, rerun_mode):
    try:
        log_step_start(run_id, pipeline_name, step_name, "silver tables", target_table, pipeline_start_time)

        orders = spark.read.table("dbo.silver_orders")
        items = spark.read.table("dbo.silver_order_items")
        products = spark.read.table("dbo.silver_products").select("product_id", "unit_price", "category", "product_name")
        customers = spark.read.table("dbo.silver_customers").select("customer_id", "country")

        fact_df = (
            orders.alias("o")
            .join(items.alias("i"), "order_id")
            .join(products.alias("p"), "product_id", "left")
            .join(customers.alias("c"), "customer_id", "left")
            .select(
                F.col("o.order_id").alias("order_id"),
                F.col("o.customer_id").alias("customer_id"),
                F.col("i.product_id").alias("product_id"),
                F.to_date(F.col("o.order_ts")).alias("order_date"),
                F.col("i.quantity").cast("int").alias("quantity"),
                F.col("p.unit_price").cast("decimal(18,2)").alias("unit_price"),
                (
                    F.col("i.quantity").cast("decimal(18,2)") *
                    F.col("p.unit_price").cast("decimal(18,2)")
                ).alias("line_amount"),
                F.col("c.country").alias("region")
            )
        )

        merge_upsert(
            fact_df,
            target_table,
            "t.order_id = s.order_id AND t.product_id = s.product_id"
        )

        count = fact_df.count()

        log_step_end(run_id, pipeline_name, step_name, "SUCCESS", "IN_PROGRESS", count)

    except Exception as e:
        log_step_end(run_id, pipeline_name, step_name, "FAILED", "FAILED", 0, str(e), datetime.now())
        raise
else:
    print(f"{step_name} skipped")

StatementMeta(, 2d49eeb2-2977-4e3d-9f06-806883a16afa, 9, Finished, Available, Finished, False)

In [13]:
# =========================================================
# 5. FACT_RETURNS (Derived Quantity from Order Items)
# =========================================================

step_name = "gold_fact_returns"
target_table = "dbo.fact_returns"

if should_run_step(run_id, pipeline_name, step_name, rerun_mode):
    try:
        log_step_start(run_id, pipeline_name, step_name, 
                       "silver_returns + silver_order_items + silver_products", 
                       target_table, pipeline_start_time)

        returns_df = spark.read.table("dbo.silver_returns")
        items_df = spark.read.table("dbo.silver_order_items")
        products_df = spark.read.table("dbo.silver_products").select("product_id", "unit_price")

        fact_df = (
            returns_df.alias("r")
            .join(items_df.alias("i"), ["order_id"], "left")
            .join(products_df.alias("p"), ["product_id"], "left")
            .select(
                F.col("r.return_id").alias("return_id"),
                F.col("r.order_id").alias("order_id"),
                F.col("p.product_id").alias("product_id"),
                F.to_date(F.col("r.return_ts")).alias("return_date"),
                F.col("i.quantity").cast("int").alias("return_qty"),
                F.col("p.unit_price").cast("decimal(18,2)").alias("unit_price"),
                (
                    F.col("i.quantity").cast("decimal(18,2)") *
                    F.col("p.unit_price").cast("decimal(18,2)")
                ).alias("return_amount")
            )
            # Data quality filters
            .filter(F.col("return_id").isNotNull())
            .filter(F.col("order_id").isNotNull())
            .filter(F.col("product_id").isNotNull())
            .filter(F.col("return_date").isNotNull())
            .filter(F.col("return_qty").isNotNull())
            .filter(F.col("return_qty") > 0)
        )

        merge_upsert(
            fact_df,
            target_table,
            "t.return_id = s.return_id"
        )

        count = fact_df.count()

        log_step_end(run_id, pipeline_name, step_name, "SUCCESS", "IN_PROGRESS", count)

    except Exception as e:
        log_step_end(run_id, pipeline_name, step_name, "FAILED", "FAILED", 0, str(e), datetime.now())
        raise
else:
    print(f"{step_name} skipped")

StatementMeta(, 2d49eeb2-2977-4e3d-9f06-806883a16afa, 15, Finished, Available, Finished, False)

In [14]:
# =========================================================
# 6. Finalize_pipeline_run
# =========================================================

finalize_pipeline_run(run_id, pipeline_name, "SUCCESS")

print("Gold layer completed successfully")
print(f"pipeline_run_id = {run_id}")

StatementMeta(, 2d49eeb2-2977-4e3d-9f06-806883a16afa, 16, Finished, Available, Finished, False)

Gold layer completed successfully
pipeline_run_id = 45e6d27f-77cb-49a0-b1ae-7eecf73012bb
